In [2]:
from common import *
from db import get_engine

engine = get_engine()

## 1. Ümumi KPI nəticələrinin hazırlanması

Bu mərhələdə Silver cədvəlindən əsas uçuş KPI-ları hesablanır. Məqsəd Power BI və reporting üçün hazır yekun nəticə yaratmaqdır.

In [3]:
gold_kpi = pd.read_sql("""
SELECT
    COUNT(*) AS TOTAL_FLIGHTS,
    SUM(CASE WHEN FLIGHT_STATUS = 'Completed' THEN 1 ELSE 0 END) AS COMPLETED_FLIGHTS,
    SUM(CANCELLED) AS CANCELLED_FLIGHTS,
    SUM(DIVERTED) AS DIVERTED_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
""", con=engine)

gold_kpi

,TOTAL_FLIGHTS,COMPLETED_FLIGHTS,CANCELLED_FLIGHTS,DIVERTED_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,999657,981515,15527,2615,18.6,4.39


#### Nəticənin Gold cədvəlinə yazılması

Hazırlanmış ümumi KPI nəticələri `gold.flight_kpi_summary` cədvəlinə yazılır.

In [4]:
gold_kpi.to_sql(
    "flight_kpi_summary",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

1

## 2. Aviaşirkətlər üzrə performans nəticələrinin hazırlanması

Bu mərhələdə aviaşirkətlər üzrə ümumi uçuş sayı, gecikmə faizi və orta çatma gecikməsi hesablanır. Məqsəd nəticəni Power BI üçün Gold layer-da hazırlamaqdır.

In [5]:
gold_airline = pd.read_sql("""
SELECT
    AIRLINE_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY AIRLINE_NAME
ORDER BY DELAY_RATE
""", con=engine)

gold_airline

,AIRLINE_NAME,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Hawaiian Airlines Inc.,13122,10.93,1.87
1,Alaska Airlines Inc.,29609,13.05,-1.06
2,Delta Air Lines Inc.,149858,13.65,0.28
3,American Airlines Inc.,124964,18.20,3.18
4,Skywest Airlines Inc.,100614,18.63,5.83
5,US Airways Inc.,34278,18.89,3.77
6,Southwest Airlines Co.,217546,19.04,4.39
7,Virgin America,10703,19.35,4.91
8,Atlantic Southeast Airlines,98165,19.64,6.66
9,United Air Lines Inc.,88596,20.54,5.33


#### Nəticənin Gold cədvəlinə yazılması

Aşağıdakı addımda hazırlanmış aviaşirkət performans nəticələri `gold.airline_performance` cədvəlinə yazılır.

In [6]:
gold_airline.to_sql(
    "airline_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

14

## 3. Aylıq performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların aylar üzrə ümumi sayı, gecikmə faizi, orta çatma gecikməsi, ləğv və diversion faizləri hesablanır.
Məqsəd nəticələri Power BI üçün hazır Gold cədvəlinə çevirməkdir.

In [7]:
gold_monthly = pd.read_sql("""
SELECT
    MONTH,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(100.0 * SUM(CANCELLED) / COUNT(*), 2) AS CANCELLATION_RATE,
    ROUND(100.0 * SUM(DIVERTED) / COUNT(*), 2) AS DIVERSION_RATE
FROM silver.flights_clean
GROUP BY MONTH
""", con=engine)

gold_monthly

,MONTH,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY,CANCELLATION_RATE,DIVERSION_RATE
0,April,83792,17.30,3.43,0.93,0.30
1,August,87623,18.60,4.51,1.01,0.30
2,December,82536,20.64,6.07,1.75,0.30
3,February,74041,23.16,8.28,4.75,0.23
4,January,80869,21.01,5.70,2.57,0.22
5,July,89974,20.67,6.21,0.95,0.29
6,June,86385,23.52,9.58,1.82,0.39
7,March,86649,19.41,5.01,2.15,0.23
8,May,85504,18.26,4.45,1.16,0.34
9,November,80361,15.17,0.88,1.00,0.22


#### Nəticənin Gold cədvəlinə yazılması

Hazırlanmış aylıq performans nəticələri `gold.monthly_performance` cədvəlinə yazılır.

In [8]:
gold_monthly.to_sql(
    "monthly_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

12

## 4. Günün vaxtına görə performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların günün müxtəlif vaxtlarına görə ümumi sayı, gecikmə faizi və orta çatma gecikməsi hesablanır.
Məqsəd hansı vaxt aralığında uçuş performansının daha zəif olduğunu Power BI-da göstərməkdir.

In [9]:
gold_departure_period = pd.read_sql("""
SELECT
    DEPARTURE_PERIOD,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY DEPARTURE_PERIOD
ORDER BY DELAY_RATE DESC
""", con=engine)

gold_departure_period

,DEPARTURE_PERIOD,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Evening,218254,25.34,9.02
1,Afternoon,368276,21.06,6.56
2,Morning,388979,13.09,0.21
3,Night,24148,9.09,-3.14


In [12]:
gold_departure_period.to_sql(
    "departure_period",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

4

## 5. Marşrutlar üzrə performans nəticələrinin hazırlanması

Bu mərhələdə origin və destination hava limanları üzrə uçuş sayı və gecikmə faizi hesablanır.
Məqsəd gecikmənin daha yüksək olduğu marşrutları Power BI üçün hazırlamaqdır.

In [10]:
gold_route = pd.read_sql("""
SELECT
    ORIGIN_AIRPORT,
    DESTINATION_AIRPORT,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, DESTINATION_AIRPORT
HAVING COUNT(*) >= 500
""", con=engine)

gold_route

,ORIGIN_AIRPORT,DESTINATION_AIRPORT,TOTAL_FLIGHTS,DELAY_RATE
0,SEA,ANC,1201,19.93
1,AUS,ATL,561,11.91
2,BHM,ATL,609,11.92
3,BNA,ATL,630,15.18
4,CAE,ATL,510,13.81
...,...,...,...,...
488,DFW,STL,507,16.02
489,SFO,SNA,808,18.18
490,CLT,TPA,543,19.93
491,ATL,TPA,1291,16.51


In [13]:
gold_route.to_sql(
    "route",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

493

## 6. Hava limanları üzrə performans nəticələrinin hazırlanması

Bu mərhələdə origin hava limanları üzrə orta taxi-out müddəti hesablanır.
Məqsəd yerüstü əməliyyatlarda daha çox vaxt itirilən hava limanlarını Power BI üçün hazırlamaqdır.

In [11]:
gold_airport = pd.read_sql("""
SELECT
    ORIGIN_AIRPORT,
    ORIGIN_AIRPORT_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(AVG(TAXI_OUT), 2) AS AVG_TAXI_OUT
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, ORIGIN_AIRPORT_NAME
""", con=engine)

gold_airport

,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,TOTAL_FLIGHTS,AVG_TAXI_OUT
0,ABI,Abilene Regional Airport,453,9.20
1,OAJ,Albert J. Ellis Airport,217,14.18
2,SPI,Abraham Lincoln Capital Airport,271,12.28
3,ALB,Albany International Airport,1364,13.50
4,ATW,Appleton International Airport,535,15.69
...,...,...,...,...
316,CRW,Yeager Airport,454,13.66
317,YUM,Yuma International Airport,351,14.10
318,WRG,Wrangell Airport,132,8.52
319,HDN,Yampa Valley Airport (Yampa Valley Regional),115,18.21


In [14]:
gold_airport.to_sql(
    "airport",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

321

In [15]:
query = text("""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'gold'
      AND TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME;
""")

with engine.connect() as connection:
    result = connection.execute(query)
    gold_tables = [row[0] for row in result]

print(f"Gold layer table count: {len(gold_tables)}")

for table in gold_tables:
    print(table)

Gold layer table count: 6
airline_performance
airport
departure_period
flight_kpi_summary
monthly_performance
route


### 7. Həftəiçi və həftəsonu performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların həftəiçi və həftəsonu performansı müqayisə edilir.
Məqsəd gecikmə göstəricilərinin gün tipinə görə dəyişib-dəyişmədiyini Gold layer üçün hazırlamaqdır.

In [ ]:
gold_weekday_weekend = pd.read_sql("""
SELECT
    IS_WEEKEND,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY IS_WEEKEND
""", con=engine)

gold_weekday_weekend

#### Nəticənin Gold cədvəlinə yazılması

In [ ]:
gold_weekday_weekend.to_sql(
    "weekday_weekend_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 8. Gecikmə səbəbləri üzrə nəticələrin hazırlanması

Bu mərhələdə gecikmiş uçuşlarda əsas gecikmə səbəblərinin orta təsiri hesablanır.
Məqsəd gecikmələrin daha çox hansı səbəblərdən yarandığını Gold layer üçün hazırlamaqdır.

In [ ]:
gold_delay_causes = pd.read_sql("""
SELECT
    ROUND(AVG(AIRLINE_DELAY), 2) AS AIRLINE_DELAY,
    ROUND(AVG(LATE_AIRCRAFT_DELAY), 2) AS LATE_AIRCRAFT_DELAY,
    ROUND(AVG(AIR_SYSTEM_DELAY), 2) AS AIR_SYSTEM_DELAY,
    ROUND(AVG(WEATHER_DELAY), 2) AS WEATHER_DELAY,
    ROUND(AVG(SECURITY_DELAY), 2) AS SECURITY_DELAY
FROM silver.flights_clean
WHERE IS_DELAYED = 1
""", con=engine)

gold_delay_causes

#### Nəticənin Gold cədvəlinə yazılması

In [ ]:
gold_delay_causes.to_sql(
    "delay_causes",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 9. Hava şəraitinin gecikmələrə təsirinin hazırlanması

Bu mərhələdə yağışlı və yağışsız hava şəraitində uçuş gecikmələri müqayisə edilir.
Məqsəd hava şəraitinin gecikmə göstəricilərinə təsirini Gold layer üçün hazırlamaqdır.

In [ ]:
gold_weather_impact = pd.read_sql("""
SELECT
    CASE 
        WHEN WEATHER_PRECIPITATION > 0 THEN 'Rain'
        ELSE 'No Rain'
    END AS WEATHER_GROUP,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
WHERE WEATHER_PRECIPITATION IS NOT NULL
GROUP BY 
    CASE 
        WHEN WEATHER_PRECIPITATION > 0 THEN 'Rain'
        ELSE 'No Rain'
    END
""", con=engine)

gold_weather_impact

#### Nəticənin Gold cədvəlinə yazılması

In [ ]:
gold_weather_impact.to_sql(
    "weather_impact",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 10. Məsafənin gecikmələrə təsirinin hazırlanması

Bu mərhələdə uzun məsafəli və digər uçuşların gecikmə göstəriciləri müqayisə edilir.
Məqsəd uçuş məsafəsinin gecikmə performansına təsirini Gold layer üçün hazırlamaqdır.

In [ ]:
gold_distance_impact = pd.read_sql("""
SELECT
    IS_LONG_DISTANCE,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(AVG(AIR_TIME), 2) AS AVG_AIR_TIME
FROM silver.flights_clean
GROUP BY IS_LONG_DISTANCE
""", con=engine)

gold_distance_impact

#### Nəticənin Gold cədvəlinə yazılması

In [ ]:
gold_distance_impact.to_sql(
    "distance_impact",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 11. Aviaşirkətlərin gecikməni bərpa etmə performansının hazırlanması

Bu mərhələdə aviaşirkətlərin uçuş zamanı başlanğıc gecikmənin nə qədərini bərpa edə bildiyi hesablanır.
Məqsəd əməliyyat səmərəliliyini Gold layer üçün hazırlamaqdır.


In [ ]:
gold_delay_recovery = pd.read_sql("""
SELECT
    AIRLINE_NAME,
    ROUND(AVG(DEPARTURE_DELAY), 2) AS AVG_DEPARTURE_DELAY,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(AVG(DEPARTURE_DELAY - ARRIVAL_DELAY), 2) AS TIME_RECOVERED
FROM silver.flights_clean
WHERE DEPARTURE_DELAY > 0
  AND ARRIVAL_DELAY IS NOT NULL
GROUP BY AIRLINE_NAME
ORDER BY TIME_RECOVERED DESC
""", con=engine)

gold_delay_recovery

#### Nəticənin Gold cədvəlinə yazılması

In [ ]:
gold_delay_recovery.to_sql(
    "delay_recovery",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

## Power BI üçün Star Schema modelinin hazırlanması

In [ ]:
silver_structure = pd.read_sql("""
SELECT
    COLUMN_NAME,
    DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver'
  AND TABLE_NAME = 'flights_clean'
ORDER BY ORDINAL_POSITION
""", con=engine)

silver_structure

### 12. Airline Dimension cədvəlinin hazırlanması

Bu mərhələdə aviaşirkətlər üçün unikal dimension cədvəli hazırlanır.
Dim Airline cədvəli Fact Flights ilə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [ ]:
dim_airline = pd.read_sql("""
SELECT DISTINCT
    AIRLINE_NAME
FROM silver.flights_clean
WHERE AIRLINE_NAME IS NOT NULL
ORDER BY AIRLINE_NAME
""", con=engine)

dim_airline.insert(
    0,
    "AIRLINE_ID",
    range(1, len(dim_airline) + 1)
)

dim_airline

#### Dim Airline cədvəlinin Gold layer-ə yazılması

In [ ]:
dim_airline.to_sql(
    "dim_airline",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 13. Origin Airport Dimension cədvəlinin hazırlanması

Bu mərhələdə uçuşların başladığı hava limanları üçün unikal dimension cədvəli hazırlanır.
Bu cədvəl Fact Flights ilə origin airport üzrə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [ ]:
dim_origin_airport = pd.read_sql("""
SELECT DISTINCT
    ORIGIN_AIRPORT,
    ORIGIN_AIRPORT_NAME
FROM silver.flights_clean
WHERE ORIGIN_AIRPORT IS NOT NULL
ORDER BY ORIGIN_AIRPORT
""", con=engine)

dim_origin_airport.insert(
    0,
    "ORIGIN_AIRPORT_ID",
    range(1, len(dim_origin_airport) + 1)
)

dim_origin_airport

#### Origin Airport Dimension cədvəlinin Gold layer-ə yazılması

In [ ]:
dim_origin_airport.to_sql(
    "dim_origin_airport",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 14. Destination Airport Dimension cədvəlinin hazırlanması

Bu mərhələdə uçuşların çatdığı hava limanları üçün unikal dimension cədvəli hazırlanır.
Bu cədvəl Fact Flights ilə destination airport üzrə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [ ]:
dim_destination_airport = pd.read_sql("""
SELECT DISTINCT
    DESTINATION_AIRPORT
FROM silver.flights_clean
WHERE DESTINATION_AIRPORT IS NOT NULL
ORDER BY DESTINATION_AIRPORT
""", con=engine)

dim_destination_airport.insert(
    0,
    "DESTINATION_AIRPORT_ID",
    range(1, len(dim_destination_airport) + 1)
)

dim_destination_airport

#### Destination Airport Dimension cədvəlinin Gold layer-ə yazılması

In [ ]:
dim_destination_airport.to_sql(
    "dim_destination_airport",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 15. Month Dimension cədvəlinin hazırlanması

Bu mərhələdə aylar üçün unikal dimension cədvəli hazırlanır.
Bu cədvəl Fact Flights ilə ay üzrə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [ ]:
dim_month = pd.read_sql("""
SELECT DISTINCT
    MONTH
FROM silver.flights_clean
WHERE MONTH IS NOT NULL
ORDER BY MONTH
""", con=engine)

dim_month.insert(
    0,
    "MONTH_ID",
    range(1, len(dim_month) + 1)
)

dim_month

#### Month Dimension cədvəlinin Gold layer-ə yazılması

In [ ]:
dim_month.to_sql(
    "dim_month",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 16. Fact Flights cədvəlinin hazırlanması

Bu mərhələdə hər uçuşu təmsil edən əsas Fact cədvəli hazırlanır.
Dimension cədvəllərinin ID-ləri Fact Flights cədvəlinə əlavə edilir və Power BI-də one-to-many əlaqələrin qurulması üçün istifadə olunur.

In [ ]:
fact_flights = pd.read_sql("""
SELECT
    a.AIRLINE_ID,
    o.ORIGIN_AIRPORT_ID,
    d.DESTINATION_AIRPORT_ID,
    m.MONTH_ID,

    f.IS_WEEKEND,
    f.DEPARTURE_PERIOD,
    f.DISTANCE,
    f.IS_LONG_DISTANCE,

    f.DEPARTURE_DELAY,
    f.ARRIVAL_DELAY,
    f.TAXI_OUT,
    f.AIR_TIME,

    f.CANCELLED,
    f.DIVERTED,
    f.IS_DELAYED,
    f.FLIGHT_STATUS,

    f.AIRLINE_DELAY,
    f.LATE_AIRCRAFT_DELAY,
    f.AIR_SYSTEM_DELAY,
    f.WEATHER_DELAY,
    f.SECURITY_DELAY,
    f.WEATHER_PRECIPITATION

FROM silver.flights_clean f

LEFT JOIN gold.dim_airline a
    ON f.AIRLINE_NAME = a.AIRLINE_NAME

LEFT JOIN gold.dim_origin_airport o
    ON f.ORIGIN_AIRPORT = o.ORIGIN_AIRPORT

LEFT JOIN gold.dim_destination_airport d
    ON f.DESTINATION_AIRPORT = d.DESTINATION_AIRPORT

LEFT JOIN gold.dim_month m
    ON f.MONTH = m.MONTH
""", con=engine)

fact_flights

#### Fact Flights cədvəlinin Gold layer-ə yazılması

In [ ]:
fact_flights.to_sql(
    "fact_flights",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

### 17. Star Schema cədvəllərinin yoxlanılması

In [ ]:
pd.read_sql("""
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'gold'
  AND TABLE_NAME IN (
      'dim_airline',
      'dim_origin_airport',
      'dim_destination_airport',
      'dim_month',
      'fact_flights'
  )
ORDER BY TABLE_NAME
""", con=engine)